# AED v3 — MobileNetV2 + log-mel/MFCC em entrada única

Objetivo: melhorar o AED e corrigir o problema de exportação TFLite usando **uma única entrada** `(128,128,3)`.

Cada canal da imagem de entrada representa:

1. log-mel;
2. MFCC redimensionado;
3. delta do log-mel.

Classes:

```text
ambiente_normal
grito_panico
choro_distress
voz_agressiva
impacto
vidro_quebrando
tiro
```

Meta operacional, não promessa: buscar `accuracy >= 0.90` no teste. Além da métrica multiclasse, o notebook mede também `perigo vs ambiente_normal`, porque é essa decisão que aciona o sistema.

Mudanças em relação ao AED v2:

- mais dados: ASVP-ESD, Gunshot, ESC-50, TUT, UrbanSound8K e dataset scream/non-scream;
- entrada única, evitando erro de ordem das entradas no TFLite;
- MobileNetV2 pré-treinada;
- avaliação por janela e por gravação;
- auditoria de falso alerta em áudio normal;
- exportação Keras, TFLite Float32/Float16/INT8;
- VAD continua separado, apenas para acionar Whisper/NLP.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install kagglehub librosa==0.11.0 soundfile scikit-learn pandas matplotlib seaborn tqdm pyyaml webrtcvad-wheels

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any
from collections import Counter
from google.colab import files
from IPython.display import Audio, display
from tqdm.auto import tqdm

import json, random, shutil, subprocess, zipfile, os, re, time
import kagglehub
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
import tensorflow as tf
import webrtcvad

from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score, accuracy_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass

## 1. Configuração

In [ ]:
@dataclass(frozen=True)
class ConfiguracaoAED:
    sr: int = 16000
    duracao: float = 2.0
    passo_treino: float = 1.0
    passo_inferencia: float = 0.5
    n_fft: int = 512
    hop_length: int = 256
    n_mels: int = 96
    n_mfcc: int = 40
    imagem: int = 128

    max_fontes_por_classe: int = 950
    max_janelas_por_fonte: int = 3
    copias_aug_treino: int = 1

    epocas_cabeca: int = 18
    epocas_fino: int = 25
    lote: int = 32
    paciencia: int = 8
    seed: int = 42

    usar_urban8k: bool = True
    usar_scream_dataset: bool = True
    usar_audio_violence_binario: bool = False

    minimo_consecutivas_alerta: int = 2
    max_falsos_alertas_por_hora_alvo: float = 30.0

CFG = ConfiguracaoAED()
CLASSES = ['ambiente_normal','grito_panico','choro_distress','voz_agressiva','impacto','vidro_quebrando','tiro']
ID = {c:i for i,c in enumerate(CLASSES)}
PERIGO = {c:(0 if c=='ambiente_normal' else 1) for c in CLASSES}

RAIZ = Path('/content/aed_v3')
DADOS = RAIZ/'datasets'
PROC = RAIZ/'processado'
MODELOS = RAIZ/'modelos'
RESULT = RAIZ/'resultados'
ENTREGA = RAIZ/'entrega'
for p in [RAIZ,DADOS,PROC,MODELOS,RESULT,ENTREGA]: p.mkdir(parents=True, exist_ok=True)
random.seed(CFG.seed); np.random.seed(CFG.seed); tf.random.set_seed(CFG.seed)
display(pd.DataFrame([asdict(CFG)]))

## 2. Baixar datasets

In [ ]:
# Kaggle
ASVP = Path(kagglehub.dataset_download('dejolilandry/asvpesdspeech-nonspeech-emotional-utterances'))
GUNSHOT = Path(kagglehub.dataset_download('emrahaydemr/gunshot-audio-dataset'))
print('ASVP:', ASVP)
print('Gunshot:', GUNSHOT)
if CFG.usar_urban8k:
    URBAN = Path(kagglehub.dataset_download('chrisfilo/urbansound8k'))
    print('UrbanSound8K:', URBAN)
else:
    URBAN = None
if CFG.usar_scream_dataset:
    try:
        SCREAM = Path(kagglehub.dataset_download('aananehsansiam/audio-dataset-of-scream-and-non-scream'))
        print('Scream/non-scream:', SCREAM)
    except Exception as e:
        SCREAM = None
        print('Falha no scream dataset, seguindo sem ele:', e)
else:
    SCREAM = None

In [ ]:
# ESC-50 e TUT Rare
URL_ESC = 'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip'
URL_TUT = 'https://zenodo.org/records/401395/files/TUT-rare-sound-events-2017-development.source_data_events.zip?download=1'
ESC_DIR = DADOS/'esc50'
TUT_DIR = DADOS/'tut_rare'

def baixar(url, dst):
    if dst.exists() and dst.stat().st_size>0: return
    subprocess.run(['wget','-c',url,'-O',str(dst)], check=True)

def extrair(zip_path, dst):
    if (dst/'.extraido').exists(): return
    with zipfile.ZipFile(zip_path) as z: z.extractall(dst)
    (dst/'.extraido').touch()

baixar(URL_ESC, DADOS/'esc50.zip'); extrair(DADOS/'esc50.zip', ESC_DIR)
baixar(URL_TUT, DADOS/'tut_eventos.zip'); extrair(DADOS/'tut_eventos.zip', TUT_DIR)
print('ESC/TUT prontos')

## 3. Construir metadados unificados

In [ ]:
EXT_AUDIO={'.wav','.mp3','.ogg','.flac','.m4a'}

def audios_em(p:Path):
    return sorted(x for x in p.rglob('*') if x.suffix.lower() in EXT_AUDIO)

def add(registros, arquivo, classe, fonte, grupo, origem='', extra=''):
    if Path(arquivo).exists() and classe in ID:
        registros.append({'arquivo':str(Path(arquivo).resolve()),'classe':classe,'fonte':fonte,'grupo':str(grupo),'origem_id':str(origem),'extra':str(extra)})

reg=[]

# ASVP: códigos usados no notebook anterior
MAPA_ASVP={'02':'ambiente_normal','04':'choro_distress','05':'voz_agressiva','06':'grito_panico'}
for a in audios_em(ASVP):
    cod=a.stem.split('_')[0].zfill(2)
    if cod in MAPA_ASVP:
        ator=a.parent.name
        add(reg,a,MAPA_ASVP[cod],'ASVP',f'asvp:{ator}',a.stem,cod)

# Gunshot Kaggle
for a in audios_em(GUNSHOT):
    add(reg,a,'tiro','GunshotKaggle',f'gunshot:{a.parent.name}:{a.stem}',a.stem)

# ESC-50
csvs=list(ESC_DIR.rglob('esc50.csv'))
if not csvs: raise RuntimeError('ESC-50 sem esc50.csv')
esc_csv=csvs[0]
esc_audio=list(ESC_DIR.rglob('audio'))[0]
esc=pd.read_csv(esc_csv)
MAPA_ESC={'crying_baby':'choro_distress','glass_breaking':'vidro_quebrando','door_wood_knock':'impacto','door_wood_creaks':'impacto','clapping':'impacto'}
NORMAIS={'rain','sea_waves','crickets','chirping_birds','water_drops','wind','footsteps','laughing','breathing','snoring','clock_tick','washing_machine','vacuum_cleaner','engine','train','airplane','helicopter','thunderstorm','siren','keyboard_typing','mouse_click','can_opening','brushing_teeth'}
for r in esc.itertuples():
    cat=str(r.category)
    classe=MAPA_ESC.get(cat, 'ambiente_normal' if cat in NORMAIS else None)
    if classe:
        add(reg,esc_audio/str(r.filename),classe,'ESC-50',f'esc50:{r.src_file}',r.filename,cat)

# TUT Rare
for a in audios_em(TUT_DIR):
    s=str(a).lower().replace('-','_').replace(' ','_')
    if any(k in s for k in ['babycry','baby_cry','babycrying']): cls='choro_distress'
    elif any(k in s for k in ['glassbreak','glass_break','glassbreaking']): cls='vidro_quebrando'
    elif any(k in s for k in ['gunshot','gun_shot','gunfire']): cls='tiro'
    else: continue
    add(reg,a,cls,'TUT_Rare',f'tut:{a.stem}',a.stem)

# UrbanSound8K
if URBAN is not None:
    csvs=list(URBAN.rglob('UrbanSound8K.csv'))
    if csvs:
        meta_u=pd.read_csv(csvs[0])
        audio_root=csvs[0].parent.parent/'audio'
        map_u={'gun_shot':'tiro','jackhammer':'impacto','drilling':'impacto','air_conditioner':'ambiente_normal','children_playing':'ambiente_normal','street_music':'ambiente_normal','engine_idling':'ambiente_normal','car_horn':'ambiente_normal','siren':'ambiente_normal','dog_bark':'ambiente_normal'}
        for r in meta_u.itertuples():
            cls=map_u.get(str(r.class_ if hasattr(r,'class_') else getattr(r,'class')))
            if cls:
                path=audio_root/f'fold{r.fold}'/r.slice_file_name
                add(reg,path,cls,'UrbanSound8K',f'urban:fold{r.fold}:{r.slice_file_name}',r.slice_file_name)
    else:
        # fallback por nome de pasta
        for a in audios_em(URBAN):
            s=str(a).lower()
            if 'gun' in s: cls='tiro'
            elif any(k in s for k in ['jackhammer','drilling']): cls='impacto'
            else: cls='ambiente_normal'
            add(reg,a,cls,'UrbanSound8K',f'urban:{a.parent.name}:{a.stem}',a.stem)

# Scream/non scream
if SCREAM is not None:
    for a in audios_em(SCREAM):
        s=str(a).lower()
        if 'non' in s or 'no_scream' in s or 'nonscream' in s:
            cls='ambiente_normal'
        elif 'scream' in s:
            cls='grito_panico'
        else:
            continue
        add(reg,a,cls,'ScreamDataset',f'screamds:{a.parent.name}:{a.stem}',a.stem)

meta=pd.DataFrame(reg).drop_duplicates(subset=['arquivo','classe'])
# limita por classe para controle de RAM e balanceamento
partes=[]
for c in CLASSES:
    p=meta[meta.classe==c].copy()
    grupos=p['grupo'].drop_duplicates().tolist(); random.Random(CFG.seed+ID[c]).shuffle(grupos)
    grupos=grupos[:CFG.max_fontes_por_classe]
    partes.append(p[p.grupo.isin(grupos)])
meta=pd.concat(partes, ignore_index=True)
meta.to_csv(RESULT/'fontes_unificadas.csv', index=False)
display(meta.groupby(['classe','fonte']).size().unstack(fill_value=0))
for c in CLASSES:
    if c not in set(meta.classe): raise RuntimeError(f'Sem dados para {c}')

## 4. Auditoria rápida por áudio

In [ ]:
for c in CLASSES:
    print('
'+'='*72, '
', c)
    amostras=meta[meta.classe==c].sample(min(2, len(meta[meta.classe==c])), random_state=CFG.seed)
    for r in amostras.itertuples():
        print(r.fonte, Path(r.arquivo).name, r.extra)
        display(Audio(r.arquivo))

In [ ]:
APROVAR_AUDITORIA = False
if not APROVAR_AUDITORIA:
    raise RuntimeError('Ouça as amostras. Depois altere APROVAR_AUDITORIA para True e continue.')

## 5. Split por grupo antes do augmentation

In [ ]:
linhas=[]
for c in CLASSES:
    p=meta[meta.classe==c].copy()
    grupos=p['grupo'].drop_duplicates().to_numpy()
    gss=GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=CFG.seed+ID[c])
    idx_train, idx_tmp = next(gss.split(grupos, groups=grupos))
    g_train=set(grupos[idx_train]); tmp=grupos[idx_tmp]
    gss2=GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=CFG.seed+99+ID[c])
    idx_val, idx_test = next(gss2.split(tmp, groups=tmp))
    g_val=set(tmp[idx_val]); g_test=set(tmp[idx_test])
    for split, gs in [('treino',g_train),('validacao',g_val),('teste',g_test)]:
        q=p[p.grupo.isin(gs)].copy(); q['split']=split; linhas.append(q)
meta_split=pd.concat(linhas, ignore_index=True)
meta_split.to_csv(RESULT/'fontes_com_split.csv', index=False)
display(meta_split.groupby(['classe','split'])['grupo'].nunique().unstack(fill_value=0))
if meta_split.groupby('grupo')['split'].nunique().gt(1).any(): raise RuntimeError('Vazamento entre splits.')

## 6. Extração de features: imagem única 128×128×3

In [ ]:
AMOSTRAS=int(CFG.sr*CFG.duracao)

def carregar_audio(path):
    y,_=librosa.load(str(path), sr=CFG.sr, mono=True)
    y=np.nan_to_num(y.astype(np.float32))
    if np.max(np.abs(y))>1: y=y/np.max(np.abs(y))
    return y

def ajustar(y):
    if len(y)<AMOSTRAS: return np.pad(y,(0,AMOSTRAS-len(y)))
    return y[:AMOSTRAS]

def selecionar_janelas(y, classe):
    if len(y)<=AMOSTRAS: return [ajustar(y)]
    passo=int(CFG.sr*CFG.passo_treino)
    j=[y[i:i+AMOSTRAS] for i in range(0,len(y)-AMOSTRAS+1,passo)]
    if len(j)<=CFG.max_janelas_por_fonte: return [ajustar(x) for x in j]
    if classe=='ambiente_normal':
        idx=np.linspace(0,len(j)-1,CFG.max_janelas_por_fonte,dtype=int)
    else:
        en=[np.sqrt(np.mean(x*x)+1e-9) for x in j]
        idx=np.argsort(en)[-CFG.max_janelas_por_fonte:]
    return [ajustar(j[int(i)]) for i in sorted(idx)]

def aug(y, rng):
    z=y.copy()
    z*=10**(rng.uniform(-9,6)/20)
    z=np.roll(z, int(rng.uniform(-0.15,0.15)*len(z)))
    if rng.random()<0.65:
        snr=rng.uniform(8,28)
        ps=np.mean(z*z)+1e-10
        pn=ps/(10**(snr/10))
        z+=rng.normal(0,np.sqrt(pn),size=z.shape)
    if rng.random()<0.25:
        rate=rng.uniform(0.92,1.08)
        try:
            zz=librosa.effects.time_stretch(z, rate=rate)
            z=ajustar(zz)
        except Exception: pass
    m=np.max(np.abs(z))
    if m>1: z=z/m
    return z.astype(np.float32)

def norm_img(x):
    x=x.astype(np.float32)
    lo,hi=np.percentile(x,1),np.percentile(x,99)
    x=np.clip((x-lo)/(hi-lo+1e-6),0,1)
    return x

def resize_mat(x, size=128):
    return cv2.resize(x, (size,size), interpolation=cv2.INTER_AREA)

import cv2

def feature_image(y):
    y=ajustar(y)
    mel=librosa.feature.melspectrogram(y=y, sr=CFG.sr, n_fft=CFG.n_fft, hop_length=CFG.hop_length, n_mels=CFG.n_mels, power=2.0)
    logmel=librosa.power_to_db(mel, ref=np.max, top_db=80)
    mfcc=librosa.feature.mfcc(S=logmel, n_mfcc=CFG.n_mfcc)
    delta=librosa.feature.delta(logmel)
    c1=resize_mat(norm_img(logmel), CFG.imagem)
    c2=resize_mat(norm_img(mfcc), CFG.imagem)
    c3=resize_mat(norm_img(delta), CFG.imagem)
    return np.stack([c1,c2,c3], axis=-1).astype(np.float32)

def processar(split):
    cache=PROC/f'{split}.npz'; meta_cache=PROC/f'{split}.csv'
    if cache.exists() and meta_cache.exists():
        z=np.load(cache); return z['X'], z['y'], pd.read_csv(meta_cache)
    parte=meta_split[meta_split.split==split]
    rng=np.random.default_rng(CFG.seed+{'treino':1,'validacao':2,'teste':3}[split])
    X=[]; y=[]; regs=[]
    for r in tqdm(parte.itertuples(), total=len(parte), desc=split):
        try:
            audio=carregar_audio(r.arquivo)
            janelas=selecionar_janelas(audio, r.classe)
        except Exception:
            continue
        for ji,j in enumerate(janelas):
            versoes=[('original',j)]
            if split=='treino':
                for k in range(CFG.copias_aug_treino): versoes.append((f'aug{k+1}', aug(j,rng)))
            for v,a in versoes:
                X.append(feature_image(a)); y.append(ID[r.classe]); regs.append({'split':split,'classe':r.classe,'grupo':r.grupo,'arquivo':r.arquivo,'janela':ji,'versao':v})
    X=np.asarray(X,dtype=np.float32); y=np.asarray(y,dtype=np.int64); regs=pd.DataFrame(regs)
    np.savez_compressed(cache, X=X, y=y); regs.to_csv(meta_cache,index=False)
    return X,y,regs

X_train,y_train,meta_train=processar('treino')
X_val,y_val,meta_val=processar('validacao')
X_test,y_test,meta_test=processar('teste')
print(X_train.shape, X_val.shape, X_test.shape)

## 7. Modelo MobileNetV2

In [ ]:
base=tf.keras.applications.MobileNetV2(input_shape=(CFG.imagem,CFG.imagem,3), include_top=False, weights='imagenet')
base.trainable=False
inp=tf.keras.Input(shape=(CFG.imagem,CFG.imagem,3), name='audio_features')
x=tf.keras.applications.mobilenet_v2.preprocess_input(inp*255.0)
x=base(x, training=False)
x=tf.keras.layers.GlobalAveragePooling2D()(x)
x=tf.keras.layers.Dropout(0.35)(x)
x=tf.keras.layers.Dense(128, activation='relu')(x)
x=tf.keras.layers.Dropout(0.25)(x)
out=tf.keras.layers.Dense(len(CLASSES), activation='softmax', name='evento')(x)
model=tf.keras.Model(inp,out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2,name='top2')])
model.summary()

## 8. Treino da cabeça e ajuste fino

In [ ]:
class MacroF1Callback(tf.keras.callbacks.Callback):
    def __init__(self, X, y): super().__init__(); self.X=X; self.y=y
    def on_epoch_end(self, epoch, logs=None):
        p=self.model.predict(self.X, batch_size=CFG.lote, verbose=0).argmax(axis=1)
        logs=logs or {}; logs['val_macro_f1']=f1_score(self.y,p,average='macro',zero_division=0)
        print(f" — val_macro_f1: {logs['val_macro_f1']:.4f}")

weights=compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=y_train)
class_weight={i:float(w) for i,w in enumerate(weights)}
callbacks=[
    MacroF1Callback(X_val,y_val),
    tf.keras.callbacks.ModelCheckpoint(MODELOS/'aed_v3_melhor.keras', monitor='val_macro_f1', mode='max', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=CFG.paciencia, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.CSVLogger(RESULT/'historico_cabeca.csv')
]
hist1=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=CFG.epocas_cabeca,batch_size=CFG.lote,class_weight=class_weight,callbacks=callbacks,verbose=1)

# ajuste fino parcial
base.trainable=True
for layer in base.layers[:-35]:
    layer.trainable=False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2,name='top2')])
callbacks2=[
    MacroF1Callback(X_val,y_val),
    tf.keras.callbacks.ModelCheckpoint(MODELOS/'aed_v3_melhor.keras', monitor='val_macro_f1', mode='max', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=CFG.paciencia, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    tf.keras.callbacks.CSVLogger(RESULT/'historico_ajuste_fino.csv')
]
hist2=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=CFG.epocas_fino,batch_size=CFG.lote,class_weight=class_weight,callbacks=callbacks2,verbose=1)
model=tf.keras.models.load_model(MODELOS/'aed_v3_melhor.keras')

## 9. Avaliação por janela e por gravação

In [ ]:
probs=model.predict(X_test,batch_size=CFG.lote,verbose=1)
pred=probs.argmax(axis=1)
print(classification_report(y_test,pred,target_names=CLASSES,digits=4))
metrics_janela={'accuracy':accuracy_score(y_test,pred), 'balanced_accuracy':balanced_accuracy_score(y_test,pred), 'f1_macro':f1_score(y_test,pred,average='macro'), 'f1_weighted':f1_score(y_test,pred,average='weighted')}
print(json.dumps(metrics_janela, indent=2))
pd.DataFrame(classification_report(y_test,pred,target_names=CLASSES,output_dict=True)).T.to_csv(RESULT/'metricas_janela.csv')
cm=confusion_matrix(y_test,pred)
plt.figure(figsize=(10,8)); sns.heatmap(cm,annot=True,fmt='d',xticklabels=CLASSES,yticklabels=CLASSES,cmap='Blues'); plt.xticks(rotation=45,ha='right'); plt.title('Matriz de confusão - janela'); plt.tight_layout(); plt.savefig(RESULT/'matriz_confusao_janela.png',dpi=160); plt.show()

# agregação por gravação: média das probabilidades por grupo
pred_df=meta_test.copy(); pred_df['real_id']=y_test
for i,c in enumerate(CLASSES): pred_df[f'p_{c}']=probs[:,i]
linhas=[]
for grupo, g in pred_df.groupby('grupo'):
    p=np.array([g[f'p_{c}'].mean() for c in CLASSES])
    real=int(g['real_id'].mode().iloc[0]); pr=int(p.argmax())
    linhas.append({'grupo':grupo,'real':real,'previsto':pr,'real_nome':CLASSES[real],'previsto_nome':CLASSES[pr],'confianca':float(p.max())})
grav=pd.DataFrame(linhas)
metrics_grav={'accuracy':accuracy_score(grav.real,grav.previsto), 'balanced_accuracy':balanced_accuracy_score(grav.real,grav.previsto), 'f1_macro':f1_score(grav.real,grav.previsto,average='macro'), 'f1_weighted':f1_score(grav.real,grav.previsto,average='weighted')}
print('Por gravação:', json.dumps(metrics_grav, indent=2))
grav.to_csv(RESULT/'predicoes_por_gravacao.csv', index=False)
pd.DataFrame([metrics_janela, metrics_grav], index=['janela','gravacao']).to_csv(RESULT/'resumo_metricas.csv')

## 10. Limiares por classe e decisão binária perigo

In [ ]:
probs_val=model.predict(X_val,batch_size=CFG.lote,verbose=0)
limiares={}
for i,c in enumerate(CLASSES):
    yb=(y_val==i).astype(int)
    best=(0.5,-1)
    for th in np.linspace(0.25,0.92,50):
        f=f1_score(yb,(probs_val[:,i]>=th).astype(int),zero_division=0)
        if f>best[1]: best=(float(th),float(f))
    limiares[c]=best[0]
pd.DataFrame([{'classe':c,'limiar':t} for c,t in limiares.items()]).to_csv(RESULT/'limiares.csv', index=False)
print(limiares)

# binário perigo vs normal
real_bin=np.array([0 if CLASSES[i]=='ambiente_normal' else 1 for i in y_test])
pred_bin=np.array([0 if CLASSES[i]=='ambiente_normal' else 1 for i in pred])
print('Binário perigo vs normal')
print(classification_report(real_bin,pred_bin,target_names=['normal','perigo'],digits=4))

## 11. Auditoria de falsos alertas em ambiente normal

In [ ]:
# normal só do teste
normais=meta_test[meta_test.classe=='ambiente_normal'].copy()
if not normais.empty:
    idx=normais.index.to_numpy()
    probs_norm=probs[idx]
    pred_norm=pred[idx]
    ativos=[]
    for p,pr in zip(probs_norm,pred_norm):
        classe=CLASSES[int(pr)]
        ativos.append(classe!='ambiente_normal' and p[int(pr)]>=limiares[classe])
    falsos=sum(ativos)
    horas=len(idx)*CFG.duracao/3600
    taxa=falsos/horas if horas>0 else 0
    auditoria={'janelas_normais':int(len(idx)), 'horas_estimadas':float(horas), 'falsos_alertas_janela':int(falsos), 'falsos_alertas_por_hora':float(taxa)}
    print(json.dumps(auditoria,indent=2))
    (RESULT/'auditoria_falsos_alertas.json').write_text(json.dumps(auditoria,indent=2), encoding='utf-8')
    print('Alvo falsos alertas/h <=', CFG.max_falsos_alertas_por_hora_alvo, 'OK:', taxa <= CFG.max_falsos_alertas_por_hora_alvo)

## 12. Exportação TFLite com uma entrada única

In [ ]:
model.save(MODELOS/'aed_v3_final.keras')
conv=tf.lite.TFLiteConverter.from_keras_model(model)
tflite=conv.convert(); (MODELOS/'aed_v3_float32.tflite').write_bytes(tflite)
conv=tf.lite.TFLiteConverter.from_keras_model(model); conv.optimizations=[tf.lite.Optimize.DEFAULT]; conv.target_spec.supported_types=[tf.float16]
tflite16=conv.convert(); (MODELOS/'aed_v3_float16.tflite').write_bytes(tflite16)

def rep_data():
    for i in range(min(300,len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]
try:
    conv=tf.lite.TFLiteConverter.from_keras_model(model); conv.optimizations=[tf.lite.Optimize.DEFAULT]; conv.representative_dataset=rep_data
    conv.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]; conv.inference_input_type=tf.int8; conv.inference_output_type=tf.int8
    int8=conv.convert(); (MODELOS/'aed_v3_int8.tflite').write_bytes(int8); print('INT8 exportado')
except Exception as e:
    print('INT8 falhou:', e)
for f in MODELOS.glob('*'):
    if f.is_file(): print(f.name, f.stat().st_size/1e6, 'MB')

## 13. Paridade Keras × TFLite

In [ ]:
def predict_tflite(path, X):
    it=tf.lite.Interpreter(model_path=str(path)); it.allocate_tensors()
    inp=it.get_input_details()[0]; out=it.get_output_details()[0]
    arr=X.astype(inp['dtype'])
    if inp['dtype'] in [np.int8, np.uint8]:
        scale,zero=inp['quantization']; arr=(X/scale+zero).round().astype(inp['dtype']) if scale else X.astype(inp['dtype'])
    preds=[]
    for i in range(len(arr)):
        it.set_tensor(inp['index'], arr[i:i+1]); it.invoke(); y=it.get_tensor(out['index'])
        if out['dtype'] in [np.int8,np.uint8]:
            scale,zero=out['quantization']; y=(y.astype(np.float32)-zero)*scale
        preds.append(y[0])
    return np.asarray(preds)

amostra=X_test[:250]
keras=model.predict(amostra,verbose=0)
for name in ['aed_v3_float32.tflite','aed_v3_float16.tflite','aed_v3_int8.tflite']:
    path=MODELOS/name
    if path.exists():
        t=predict_tflite(path,amostra)
        acc=float(np.mean(t.argmax(axis=1)==keras.argmax(axis=1)))
        diff=float(np.max(np.abs(t-keras)))
        print(name, 'concordancia_argmax=',acc, 'max_diff=',diff)

## 14. Teste em áudio externo e VAD paralelo

In [ ]:
enviados=files.upload()
auds=[n for n in enviados if Path(n).suffix.lower() in EXT_AUDIO]
if len(auds)!=1: raise RuntimeError('Envie exatamente um áudio.')
TESTE=Path(auds[0]).resolve()
display(Audio(str(TESTE)))

def janelas_deslizantes(y, passo_seg):
    passo=int(CFG.sr*passo_seg)
    if len(y)<=AMOSTRAS: return [(0,ajustar(y))]
    return [(i/CFG.sr, y[i:i+AMOSTRAS]) for i in range(0,len(y)-AMOSTRAS+1,passo)]

def inferir(path):
    y=carregar_audio(path); js=janelas_deslizantes(y, CFG.passo_inferencia)
    X=np.asarray([feature_image(j) for _,j in js], dtype=np.float32)
    pr=model.predict(X,verbose=0)
    linhas=[]
    for (t,_),p in zip(js,pr):
        cid=int(p.argmax()); cls=CLASSES[cid]; conf=float(p[cid]); ativo=cls!='ambiente_normal' and conf>=limiares[cls]
        linhas.append({'inicio':t,'fim':t+CFG.duracao,'classe':cls,'confianca':conf,'limiar':limiares[cls],'ativo':ativo})
    df=pd.DataFrame(linhas)
    cons=0; last=None; alertas=[]
    for r in df.itertuples():
        if r.ativo and r.classe==last: cons+=1
        elif r.ativo: cons=1
        else: cons=0
        last=r.classe if r.ativo else None
        alertas.append(cons>=CFG.minimo_consecutivas_alerta)
    df['alerta_temporal']=alertas
    return df

tl=inferir(TESTE); display(tl.sort_values('confianca',ascending=False).head(20)); tl.to_csv(RESULT/'timeline_audio_externo.csv',index=False)

# VAD não bloqueia AED; só decide se Whisper/NLP deve rodar
def vad_info(path):
    y=carregar_audio(path); pcm=(np.clip(y,-1,1)*32767).astype(np.int16)
    vad=webrtcvad.Vad(2); frame_ms=30; n=int(CFG.sr*frame_ms/1000); dec=[]
    for i in range(0,len(pcm)-n+1,n): dec.append(vad.is_speech(pcm[i:i+n].tobytes(), CFG.sr))
    return {'frames':len(dec),'frames_fala':int(sum(dec)),'proporcao_fala':float(np.mean(dec)) if dec else 0}
print(vad_info(TESTE))

## 15. Pacote final

In [ ]:
if ENTREGA.exists(): shutil.rmtree(ENTREGA)
ENTREGA.mkdir(parents=True)
for f in MODELOS.glob('*'):
    if f.is_file(): shutil.copy2(f, ENTREGA/f.name)
for f in RESULT.glob('*'):
    if f.is_file(): shutil.copy2(f, ENTREGA/f.name)
config={'configuracao':asdict(CFG),'classes':CLASSES,'limiares':limiares,'metricas_janela':metrics_janela,'metricas_gravacao':metrics_grav}
(ENTREGA/'configuracao_aed_v3.json').write_text(json.dumps(config,ensure_ascii=False,indent=2), encoding='utf-8')
zip_final=shutil.make_archive('/content/aed_v3_entrega','zip',root_dir=str(ENTREGA))
print('ZIP:', zip_final)
files.download(zip_final)

## Nota honesta

Se a acurácia multiclasse não chegar a 90%, não force a métrica. Verifique principalmente `impacto`, `grito_panico` e `voz_agressiva`, que eram os pontos fracos do v2. O alvo mínimo mais realista para a primeira versão embarcada é: alta revocação para eventos perigosos + baixa taxa de falso alerta por hora.